[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C18/E18-1_Resistance_Traction_Composite.ipynb)

# Résistance d'un composite : une étude de régression de bout en bout

## Le contexte

Un fabricant de plaques en matériau composite relève, pour chaque plaque, huit réglages du procédé de stratification, puis mesure sa résistance à la traction. Le bureau d'études voudrait prédire cette résistance avant l'essai, pour écarter les réglages qui donneront une plaque trop faible. Les 20 640 plaques mesurées forment une base assez grande pour mener une étude complète : lire les données, les préparer, comparer des modèles, puis annoncer un score.

La grandeur à prédire est une résistance en MPa : c'est un problème de régression.

Défi : prédire `resistance_traction_MPa` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est une régression linéaire sur les huit réglages mis à l'échelle ; il atteint un R² de 0,591. Un modèle sans apprentissage, qui répond toujours la moyenne, donne -0,000, et une droite sur le seul réglage le plus corrélé 0,471. À vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Resistance_Traction_Composite.xlsx`, dans le dossier `donnees` du dépôt : 20 640 plaques, 9 colonnes. Une ligne est une plaque.

| Colonne | Signification | Unité |
|---|---|---|
| `ratio_fibres_matrice` | ratio de fibres sur matrice | sans unité déclarée |
| `temperature_cuisson` | température de cuisson | sans unité déclarée |
| `pression_compactage` | pression de compactage | sans unité déclarée |
| `duree_polymerisation` | durée de polymérisation | sans unité déclarée |
| `taux_humidite_resine` | taux d'humidité de la résine | sans unité déclarée |
| `densite_fibres` | densité des fibres | sans unité déclarée |
| `vitesse_refroidissement` | vitesse de refroidissement | sans unité déclarée |
| `taux_additifs` | taux d'additifs | sans unité déclarée |
| `resistance_traction_MPa` | cible : résistance à la traction, écrêtée à 700 | MPa |

Origine : jeu public transformé. Les huit variables d'un jeu public sont réinterprétées en réglages de procédé, la grandeur à prédire devenant la résistance, écrêtée à 700 MPa. Il enseigne une méthode, et aucune conclusion de matériau n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides, ses types et la plage de la cible : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Resistance_Traction_Composite.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Resistance_Traction_Composite.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "resistance_traction_MPa"
print("dimensions :", df.shape)
print("cases vides :", int(df.isna().sum().sum()))
print("types :", df.dtypes.astype(str).value_counts().to_dict())
print(df[cible].describe().round(1).to_dict())
a_700 = (df[cible] >= 700).mean()
print("part des plaques écrêtées à 700 MPa :", round(100 * a_700, 1), "%")
df.head(3)

Lecture du résultat : 20 640 plaques, 9 colonnes numériques, aucune case vide. La résistance va de 215 à 700 MPa, avec une médiane de 380 MPa. 4,8 % des plaques sont écrêtées à 700 MPa : la mesure s'arrête à ce plafond, et un modèle ne pourra pas dépasser ce qu'il n'a jamais vu.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Resistance_Traction_Composite.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des plaques est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Tout ce qui suit et qui regarde la cible (corrélations, comparaisons de modèles) se lit sur l'apprentissage seul.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "plaques")
print("test :", len(X_test), "plaques")

### Étape 3 : explorer l'apprentissage

Trois lectures sur `X_train` et `y_train` : la corrélation de chaque réglage avec la résistance, l'asymétrie de chaque réglage (une valeur proche de 0 signale une distribution symétrique, une valeur forte une queue étirée), et le nombre de valeurs hors des bornes de Tukey.

In [ ]:
correlations = X_train.corrwith(y_train).sort_values(key=abs, ascending=False)
asymetrie = X_train.skew()
q1, q3 = X_train.quantile(0.25), X_train.quantile(0.75)
hors_bornes = ((X_train < q1 - 1.5 * (q3 - q1)) | (X_train > q3 + 1.5 * (q3 - q1))).sum()
bilan = pd.DataFrame({"corrélation": correlations, "asymétrie": asymetrie, "hors bornes": hors_bornes}).loc[correlations.index]
print(bilan.round(2).to_string())

Lecture du résultat : un seul réglage porte l'essentiel du signal linéaire, `ratio_fibres_matrice`, avec une corrélation de 0,69 ; les sept autres sont à moins de 0,16 en valeur absolue. 4 réglages ont une queue très étirée (asymétrie supérieure à 3 en valeur absolue) : l'asymétrie de `densite_fibres` atteint 85, et ils ont de nombreuses valeurs hors des bornes de Tukey. Ce sont des relevés rares et extrêmes, pas forcément des erreurs : on les garde, mais on retiendra qu'un modèle linéaire en est sensible.

### Étape 4 : construire le modèle de départ

Le modèle de départ est une régression linéaire sur les huit réglages, chacun ramené à une moyenne nulle et à un écart type de 1. Le pipeline apprend sur `X_train` seul, et un second modèle témoin n'utilise que le réglage le plus corrélé.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit des plaques qu'il n'a jamais vues. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en MPa, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Deux témoins situent le résultat : la moyenne de l'apprentissage, et la droite sur un seul réglage.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
score_une_variable = r2_score(y_test, une_variable.predict(X_test[[meilleur]]))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.1f} MPa")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.1f} MPa")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")
print(f"R² d'une droite sur {meilleur} seul : {score_une_variable:.3f}")

Lecture du résultat : le R² est de 0,591 et l'erreur absolue moyenne de 53,0 MPa, pour une résistance moyenne de 406 MPa sur la base de test. Le témoin qui répond la moyenne est à -0,000 ; la droite sur le seul réglage `ratio_fibres_matrice` explique 0,471, et ajouter les sept autres réglages en linéaire ne gagne que 0,120. C'est ce chiffre, mesuré sur des plaques que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : comparer des modèles sans toucher au test

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première compare, par validation croisée sur l'apprentissage seul, la droite, une régression ridge sur les réglages et leurs carrés, une droite sur les réglages transformés par un logarithme (pour tasser les queues étirées), une forêt aléatoire et un gradient boosting. La seconde mesure le modèle retenu une seule fois sur le test.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : en validation croisée, la droite atteint 0,608, la droite sur réglages transformés 0,656, le ridge avec carrés 0,391, la forêt 0,804 et le gradient boosting 0,837 : le modèle retenu est « gradient boosting par histogrammes ». Mesuré une seule fois sur le test, il donne un R² de 0,848, soit 0,257 de plus que le modèle de départ. Les carrés ne sont pas fiables sur des réglages à queue étirée : le ridge est le moins bon des cinq modèles. Le logarithme aide un peu la droite ; ce qui lui manque surtout, ce sont des effets qui se combinent, que les arbres capturent. L'écrêtage à 700 MPa reste une limite : le modèle y commet une erreur moyenne de 61 MPa, car il n'a jamais vu de résistance au-dessus du plafond.

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Une étude de bout en bout suit toujours le même ordre : lire la forme du fichier, mettre le test de côté, explorer l'apprentissage, préparer, comparer, mesurer une fois sur le test.
2. Un seul réglage, `ratio_fibres_matrice`, porte 0,471 du R² ; ajouter les autres en linéaire ne donne que 0,120 de plus.
3. Comparer en validation croisée sur l'apprentissage, jamais sur le test : « gradient boosting par histogrammes » gagne avec 0,837, contre 0,608 pour la droite.
4. Mesuré une seule fois sur le test, le modèle retenu donne 0,848. Le plafond de mesure à 700 MPa borne ce que le modèle sait prévoir.